<center><b style="font-size:1.17em;">Simulations</b></center>
<div style="font-size:6px; line-height:8px;">&nbsp;</div>
<center>
Subthreshold clustered <strong>glutamatergic</strong> input and fixed slow <strong>GABA<sub>A</sub></strong> extrasynaptic inputs
</center>
GABA simulation trails onset of glutamatergic activity by 20 ms.  


$\sum g_{\mathrm{GABA}}$ for each simulation is the same as trials in sim241x.
`
- **2510** – simultaneous inputs slow GABA receptors; vary nGABA 0 - 1920 
- **2511** – suprathreshold clustered **glutamatergic** input + simultaneous slow GABA receptors; vary nGABA   
- **2512** – same as 2510 but under voltage-clamp conditions with no point errors (Rs = 1 × 10⁻⁹)  
- **2513** – same as 2510 but under voltage-clamp conditions and Rs = 30 MΩ  
- **2514** – same as 2510 but block fast voltage-gated Na⁺ channels  
- **2515** – same as 2511 but block fast voltage-gated Na⁺ channels  
- **2516** – same as 2510 but block select K⁺ channels in a cell type–specific manner  
- **2517** – same as 2511 but block select K⁺ channels in a cell type–specific manner  

- **25120** – same as 2512 but under voltage-clamp conditions with no point errors (Rs = 1 × 10⁻⁹) and perfect space clamp

Simulations **2518** and **2519** run for iSPN only (this condition reduces **Threshold**  see definition (see below).

---

**Channel-specific blocking rules**

- If **dSPN** → block **KCNQ only**  
- If **iSPN** → block **KCNQ + Kir**
- If **iSPN** → block **KCNQ + Kir + Kaf**

---

Run with **cell type** set to either **ispn** or **dspn**.

**Threshold** is the minimum number of clustered **glutamatergic inputs** that consistently evoke upstates individually at **all four sites**.

In **dSPN** and **iSPN**, 15 and 14 **clustered glutamatergic inputs** are the minimum required to elicit an upstate at each site.

In **dSPN** and **iSPN** and reduced K+ conditions (**KCNQ only** and **KCNQ + Kir**, respectively), an upstate is observed with **15 and 14 clustered glutamatergic inputs** for either cell types, respectively.

In iSPN and reduced K+ conditions (**KCNQ + Kir + Kaf**), an upstate is observed with 13 clustered glutamatergic inputs.

Both **dSPN** and **iSPN** firing are unaffected by slow **GABAergic** synaptic input.

With **50 % K⁺ reduction**, minimum number of clustered **glutamatergic inputs** that consistently evoke upstates individually at **all four sites** is 15 (KCNQ only) and 14 (KCNQ + Kir) for dSPN and iSPN, respectively.

<div align="center">

### Summary

</div>

- slow GABAergic input has small effect on suprathreshold upstate / firing iSPN
- Same glutamatergic input fires more APs with lowered KCNQ & Kir (iSPN)
- K+ block has no effect on iSPNs

In [ ]:
# choose sim and cell_type
sim = '2510' 
cell_type='ispn'

In [ ]:
############################################## SETTINGS ##############################################  
# settings
model = 3
current_step = False

# import default settings
import os, sys

cwd = os.getcwd()

if os.path.exists(os.path.join(cwd, 'settings.py')):
    main_dir = cwd
else:
    main_dir = os.path.abspath(os.path.join(cwd, '..'))

os.chdir(main_dir)
sys.path.insert(0, main_dir)

%run -i settings.py

record_3D = True
record_3D_mechs = True
record_3D_Ca = True
record_Ca = False
record_currents = True
record_mechs = True
record_path_dend = True 
record_path_spines = False   

if sim in ['2510', '2516', '2518']:
    record_3D_impedance = True   

record_branch = False

if any(x in sim for x in ('2510', '2512', '2513', '2514', '2516', '2518')):
    glut = False
else:
    glut = True
 
glut_frequency = 1000 # every 10ms
AMPA = True
NMDA = True

gaba = True
# gaba_tau1 = 9
# gaba_tau2 = 180

print("model:", model)
print("mechs:", mechs) 

mechs2record = ['kaf', 'kas', 'kir', 'kcnq', 'bk', 'sk']

if any(x in sim for x in ('2512','2513','25120',)):
    voltage_clamp = True
    
if any(x in sim for x in ('25120',)):
    Ra=1.59e-10 # for silver wire for perfect space clamp

if voltage_clamp:
    scale_factors = {'naf': 0.1, 'nap': 0.1, 'kdr': 0.5, 'kas': 0.5, 'kir': 0.5, 'kcnq': 0.25}
    
    if any(x in sim for x in ('2513',)):
        Rs = 30

if any(x in sim for x in ('2514','2515',)):
    scale_factors = {'naf': 0}

if compare_last_digit(sim, 6) or compare_last_digit(sim, 7):
    if cell_type == 'dspn':
        scale_factors = {'kcnq': 0.5}
    else:
        scale_factors = {'kcnq': 0.5, 'kir': 0.5}

if compare_last_digit(sim, 8) or compare_last_digit(sim, 9):
    if cell_type == 'dspn':
        raise SystemExit("dSPN simulation invalid.")
    else:
        scale_factors = {'kcnq': 0.5, 'kir': 0.5, 'kaf': 0.5}
    v_init = -85.4087
    

# slow GABA
gaba_tau1 = 9
gaba_tau2 = 180

# each sim can produce a plot along path to soma
Nsim_plot = True
Nsim_save = False

# show overall summary plot and save
showplot = True 
save = False
dt = 0.025
sim_time = 600

deltat = dt * ds

In [ ]:
# select dendritic locations; use dend2remove1 for locations that won't be eliminated when dendrites removed
from analysis_functions import *

# dend2remove1 = ['dend[55]', 'dend[19]', 'dend[42]']
dend2remove1 = None

cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove1)
cell_coordinates = []

for sec in cell.somalist:
    h('access ' + sec.name())
    x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
    cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])

# Setup for dendritic sections
for sec in cell.dendlist:
    for seg in sec:
        x, y, z, diam = interpolate_3d(sec, seg.x)
        cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])


cell_coordinates = np.array(cell_coordinates, dtype=object)

width=600
height=600
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=0.8, color='grey', height=height, width=width)

fig_morphology.show()


In [ ]:
# chose multiple input sites for upstate generation ~ 157.5 um from the soma
cell_coordinates = np.array(cell_coordinates, dtype=object)

# Target distance
dist_target = 157.5

# Prepare an empty list to store results
closest_points = []

# Iterate over unique section names
for sec_name in np.unique(cell_coordinates[:, 0]):
    # Filter rows for the current `sec.name()`
    sec_rows = cell_coordinates[cell_coordinates[:, 0] == sec_name]
    
    # Extract distances for the current section
    distances = sec_rows[:, 5].astype(float)
    
    # Check if the target distance is within the range of this section's distances
    if distances.min() <= dist_target <= distances.max():
        # Calculate absolute differences from the target
        abs_diff = np.abs(distances - dist_target)
        
        # Find the index of the minimum difference
        min_index = np.argmin(abs_diff)
        
        # Append the closest row to the results
        closest_points.append(sec_rows[min_index])

# Convert the results to a numpy array for convenience
# Nsites = 5 # sites to generate upstates

synapses_ = np.array(closest_points, dtype=object)
selected_dendrites_df = pd.DataFrame(synapses_)

# # if random then do this:
# selected_dendrites_df = selected_dendrites_df.sample(frac=1, random_state=316309).reset_index(drop=True)
# dend_glut_full = selected_dendrites_df.iloc[0:Nsites][0].tolist()
# glut_locations_full = selected_dendrites_df.iloc[0:Nsites][1].tolist()

if cell_type == 'dspn':
    dend_glut_full = ['dend[15]', 'dend[52]', 'dend[47]', 'dend[21]'] # dend_glut_full = ['dend[15]', 'dend[52]', 'dend[28]', 'dend[47]', 'dend[57]']
    num_gluts = 15
elif cell_type == 'ispn':
    dend_glut_full = ['dend[18]', 'dend[33]', 'dend[5]', 'dend[11]']
    num_gluts = 14

if compare_last_digit(sim, 8) or compare_last_digit(sim, 9):
    num_gluts = 13

Nsites = len(dend_glut_full)

glut_locations_full = selected_dendrites_df.set_index(0).loc[dend_glut_full, 1].tolist()

dend_glut = [dend for dend in dend_glut_full for _ in range(num_gluts)]

glutamate_locations=[]
for dend, loc in zip(dend_glut_full, glut_locations_full):
    result = spine_locator(cell_type=cell_type, specs=specs, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, 
                           dend_glut=[dend], num_gluts=num_gluts, method=method, dend2remove=dend2remove, rel_x=loc)
    glutamate_locations.extend(result)

# generate rel_glut_onsets
dstep1 = int(1/glut_frequency *1e3)

if dstep1 > 0:
    rel_glut_onsets = list(range(0, num_gluts * dstep1, dstep1))
else:
    rel_glut_onsets = 0 * num_gluts  # num_gluts repetitions of glut_time

# Nsites is number of separate upstate locations
if Nsites > 1:
    rel_glut_onsets = rel_glut_onsets * Nsites
    num_gluts = num_gluts * Nsites

In [ ]:
sims251x_prefixes = ['2510', '2511', '2512', '2513', '2514', '2515', '2516', '2517', '2518', '2519', '25120']
sims251x = [f'{prefix}{letter}' for prefix in sims251x_prefixes for letter in 'abcdefghij']

sim_root = ''.join(c for c in sim if c.isdigit())

if sim_root in sims251x_prefixes:
    timing_range = list(range(150,160,10))
    Nsims = len(timing_range)
    

In [ ]:
# select synapses at random to sample from
from analysis_functions import *

dend2remove1 = ['dend[55]', 'dend[19]', 'dend[42]']
dend2remove1 = None

cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove1)
cell_coordinates = []

for sec in cell.somalist:
    h('access ' + sec.name())
    x, y, z, diam = interpolate_3d(sec, 0.5)  # Use 0.5 to refer to the center of the section
    cell_coordinates.append([sec.name(), 0.5, x, y, z, h.distance(0.5, sec=sec), diam])

# Setup for dendritic sections
for sec in cell.dendlist:
    for seg in sec:
        x, y, z, diam = interpolate_3d(sec, seg.x)
        cell_coordinates.append([sec.name(), seg.x, x, y, z, h.distance(seg.x, sec=sec), diam])


cell_coordinates = np.array(cell_coordinates, dtype=object)

width = 600
height = 600
fig_morphology = morphology_plot(cell_coordinates, dend_tree, lwd=0.8, color='grey', height=height, width=width)

fig_morphology.show()

# choose inputs
cell_coordinates_df = pd.DataFrame(cell_coordinates, columns=['section', 'rel.x', 'x', 'y', 'z', 'dist', 'diam'])
dend_df = cell_coordinates_df[cell_coordinates_df.iloc[:, 0] != 'soma[0]']

# sample N=6160 rows WITH replacement
sampled_df = dend_df.sample(n=6160, replace=True, random_state=2)
sampled_df = sampled_df.sort_index()
indices = sampled_df.index.to_list()
N_GABA_total = len(indices)

dend_gaba_full = cell_coordinates_df.loc[indices, 'section'].tolist()
gaba_locations_full = cell_coordinates_df.loc[indices, 'rel.x'].tolist()

pairs = list(zip(dend_gaba_full, gaba_locations_full))

random.seed(2) 
random.shuffle(pairs)
dend_gaba_full, gaba_locations_full = zip(*pairs)

# convert back to lists
dend_gaba_full = list(dend_gaba_full)
gaba_locations_full = list(gaba_locations_full)

    

In [ ]:
print(dend_gaba_full[0], gaba_locations_full[0]) 

In [ ]:
# Determine which list 'sim' belongs to
sim_list=None
sim_list = [s for s in sims251x if s.startswith(sim_root) and len(s) == len(sim_root) + 1]
    
if voltage_clamp:
    iClamp = []
    vdend_Clamp = []
else:
    soma_v_master = []
    dend_v_master = []
  
N_GABA_fast_dict = {
    'a': 0,     'b': 4,
    'c': 8,     'd': 12,
    'e': 16,    'f': 20,
    'g': 24,    'h': 32,
    'i': 40,    'j': 48,
}  

g_GABA = round(25 * 1e-6, 8) # 25 pS to uS
scale = 0.001 / g_GABA

N_GABA_dict =  {k: int(v * scale) for k, v in N_GABA_fast_dict.items()}

timing = 150
stim_time = 170
sim_time = 780 

if sim_list:
    for sim in tqdm(sim_list):
        sim_letter = ''.join(c for c in sim if c.isalpha())
        
        # last glutamate synapse in train
        record_dendrite = dend_glut[int(num_gluts / Nsites)-1]
        record_location = [glutamate_locations[int(num_gluts / Nsites)-1]]
            
        num_gabas = get_(sim=sim, _dict=N_GABA_dict)
        
        if voltage_clamp:
            if num_gabas ==0 :
                num_gabas = 1
                gaba = False
            else:
                gaba = True 
            
            dend_gaba = dend_gaba_full[0:num_gabas]
            gaba_locations = gaba_locations_full[0:num_gabas]
            rel_gaba_onsets = np.repeat(0, num_gabas)   
            
        else:
            if g_GABA !=0 :
                dend_gaba = dend_gaba_full[0:num_gabas]
                gaba_locations = gaba_locations_full[0:num_gabas]
                rel_gaba_onsets = np.repeat(0, num_gabas)   
                gaba = True # default is False

        
        # to store sim-generated variables
        data_dict = {
                'i_recording_site': [],  
                'v_recording_site': [], 
                'v_all_3D': {},
                'Ca_all_3D': {},
                'imp_all_3D': {},
                'i_mechs_3D': {},
                'v_dend_tree': {},
                'v_spine_tree': {},
                'v_branch': {},
                'vsoma': [],
                'vdend': [],
                'v_dend_activated': {},
                'vspine': [],
                'v_spine_activated': {},
                'Ca_soma': [],
                'Ca_dend': [],
                'Ca_spine': [],
                'timing': [],
                'dists': [],
                'dists_spine': [],
                'i_mechs_dend': {},
                'i_mechs_dend_tree': {},
                'i_mechs_spine_tree': {},
                'i_ampa': {},
                'i_nmda': {},
                'i_gaba': {},
                'g_gaba': {},
                'record_dists':[],
                'record_spine':[],
                'spine_dist': [],
                'capacitance': [],
                'timestamp': {}
                }


        # get coordinates for all unique segments within all sections
        cell, spines, dend_tree = cell_build(cell_type=cell_type, specs=specs, addSpines=True, spine_per_length=spine_per_length, frequency=frequency, d_lambda=d_lambda, verbose=False, dend2remove=dend2remove)

        # scale conductances
        if scale_factors is not None:
            printed = set()  # track which mechanisms have already been reported
            for sec in h.allsec():
                for seg in sec:
                    for mech, sf in scale_factors.items():
                        if hasattr(seg, mech):
                            getattr(seg, mech).sf = sf
                            if mech not in printed:
                                if mech in ['cal12', 'cal13', 'can', 'car', 'cav32', 'cav33']:
                                    print(f"permeability (cm/s) scaled: '{mech}' scaling factor = {sf}")
                                else:
                                    print(f"conductance (S/cm²) scaled: '{mech}' scaling factor = {sf}")
                                printed.add(mech)
 
        mechs=['pas', 'kdr', 'naf', 'kaf', 'kas', 'kir', 'kcnq', 'cal12', 'cal13', 'can', 'car', 'cav32', 'cav33', 'sk', 'bk']
        mech_distr_3D = record_mechs_distr_3D(cell=cell, mechs=mechs)

        # set these to False is they are not already assigned
        glut_reversal = globals().get('glut_reversal', 0) 
        vary_location = globals().get('vary_location', False)  
        dend_glut_range = globals().get('dend_glut_range', None)  
        vary_gaba_location = globals().get('vary_gaba_location', False)   
        
        # rec_all_path = globals().get('rec_all_path', False)  # default is False
        # if False records all v from spine / dendrite to soma
        # if True records at all unique sites including those distal to the synapse

        # determine if axoshaft or axospinous glutamatergic synapse
        axoshaft = globals().get('axoshaft', False)
        axospine = not axoshaft

        if model == 1 and not vary_location:
            glutamate_locations = sorted(glutamate_locations, reverse=True)

        metadata = {}
        # Add fixed variables to metadata
        for name in variable_names:
            try:
                if name not in metadata:
                    metadata[name] = eval(name)
            except NameError:
        #         print(f"Variable {name} not found!") # no need to print not found variables
                continue

        # syn_reversals(cell_type, specs, spine_per_length, frequency, d_lambda, dend_glut, glut_reversal, glutamate_locations, num_gluts, dend_gaba, gaba_reversal, gaba_locations, num_gabas, sim_time, dt=0.025, v_init=-84)


        # Common parameters
        common_params = {
            'cell_type': cell_type, 
            'specs': specs, 
            'spine_per_length':spine_per_length,
            'frequency': frequency,
            'd_lambda': d_lambda,
            'glut_reversal': glut_reversal,
            'num_gluts': num_gluts,
            'gaba_reversal': gaba_reversal,
            'num_gabas': num_gabas,
            'sim_time': stim_time,
            'dt': dt,
            'v_init': v_init,
            'dend2remove': dend2remove,
            'neck_dynamics': neck_dynamics,
            'soma_diameter': soma_diameter
        }

        if not vary_location:
            syn_reversals_params = {
                'dend_glut': dend_glut,
                'glutamate_locations': glutamate_locations,
                'dend_gaba': dend_gaba,
                'gaba_locations': gaba_locations,
                **common_params
            }
            glut_reversals, gaba_reversals = syn_reversals(**syn_reversals_params)

        else:
            syn_reversals_params = {
                **common_params
            }
            if vary_gaba_location:
                syn_reversals_params.update({
                    'dend_glut': dend_glut,
                    'glutamate_locations': glut_locations,
                    'dend_gaba': dend_gaba_range,
                    'gaba_locations': gaba_locations_range,
                })
                glut_reversals, gaba_reversals_range = syn_reversals(**syn_reversals_params)
                gaba_reversals_range = gaba_reversals_range * len(gaba_locations_range)
            else:
                syn_reversals_params.update({
                    'dend_glut': dend_glut_range,
                    'glutamate_locations': glut_locations_range,
                    'dend_gaba': dend_gaba,
                    'gaba_locations': gaba_locations,
                })
                glut_reversals_range, gaba_reversals = syn_reversals(**syn_reversals_params)
                glut_reversals_range = glut_reversals_range * len(glut_locations_range)


        # perform Nsim simulations 
        for ii in range(Nsims): # for ii in tqdm(list(range(0,6))):
            # time stamp date
            time = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')

            sim_image_path = None
            if Nsim_save:
                sim_image_path = 'downsample/{}/model{}/{}/images/sim{}/{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim_root, sim_letter)
                os.makedirs(sim_image_path, exist_ok=True)
 
            if vary_gaba_time:
                glut_time = stim_time
                gaba_time = timing
            else:
                glut_time = timing
                gaba_time = stim_time
            if vary_location:
                if vary_gaba_location: 
                    dend_gaba = [dend_gaba_range[ii]]
                    gaba_reversals = [gaba_reversals_range[ii]] if num_gabas == 1 else gaba_reversals_range[ii]

                else:
                    dend_glut = [dend_glut_range[ii]]
                    glut_reversals = [glut_reversals_range[ii]] if num_gluts == 1 else glut_reversals_range[ii]

                record_dendrite = record_dends[ii]
                glutamate_locations = [glut_locations_range[ii]]
                gaba_locations = [gaba_locations_range[ii]]
                record_location = [record_locs[ii]]

            if tonic:
                gbar_gaba = gbar_gaba_range[ii]
                print('tonic GABA ', gbar_gaba)
            else:
                gbar_gaba = 0

            protocol = 'Nsim{}'.format(ii)
            
            print(f"glut onset: {glut_time} ms; simultaneous gaba activation: {gaba_time} ms")
            
            # Run sim
            i_recording_site, v_recording_site, v_all_3D, Ca_all_3D, imp_all_3D, i_mechs_3D, vspine, v_spine_activated, vdend, \
            v_dend_activated, vsoma, v_dend_tree, v_spine_tree, Ca_spine, Ca_dend, Ca_soma, \
            i_mechs_dend, i_mechs_dend_tree, i_mechs_spine_tree, v_branch, zdend, ztransfer, \
            ampa_currents, nmda_currents, gaba_currents, gaba_conductances, record_dist, \
            record_spine, spine_dist, cap, dt, burn_time, start_time = \
            msNEURONsim(sim_time = sim_time, 
                     stim_time = stim_time,
                     baseline = baseline,
                     glut = glut, 
                     glut_frequency = glut_frequency, 
                     glutamate_locations = glutamate_locations, 
                     glut_reversals = glut_reversals,
                     glut_time = glut_time, 
                     num_gluts = num_gluts, 
                     dend_glut = dend_glut, 
                     g_AMPA = g_AMPA,
                     ratio = ratio,
                     gaba = gaba, 
                     gaba_frequency = gaba_frequency, 
                     gaba_locations = gaba_locations, 
                     gaba_reversals = gaba_reversals,
                     gaba_time = gaba_time, 
                     g_GABA = g_GABA, 
                     num_gabas = num_gabas, 
                     dend_gaba = dend_gaba, 
                     specs = specs, 
                     scale_factors = scale_factors, 
                     gaba_tau1 = gaba_tau1,
                     gaba_tau2 = gaba_tau2,
                     rel_gaba_onsets = rel_gaba_onsets, 
                     rel_glut_onsets = rel_glut_onsets,
                     frequency = frequency,
                     d_lambda = d_lambda,
                     dend2remove = dend2remove,
                     v_init = v_init,
                     AMPA = AMPA,
                     NMDA = NMDA,
                     physiological = physiological,
                     timing_range = timing_range,
                     add_noise = add_noise,
                     beta = beta,                   
                     B = B,                      
                     add_sine = add_sine, 
                     axoshaft = axoshaft,
                     cell_type = cell_type,
                     current_step = current_step,
                     step_current = step_current,
                     step_duration = step_duration,
                     step_start = step_start,
                     holding_current = holding_current,
                     add_ramp = add_ramp,
                     ramp_amplitude = ramp_amplitude,
                     Cm = Cm,
                     Ra = Ra,
                     spine_per_length = spine_per_length,
                     spine_neck_diam = spine_neck_diam,
                     spine_neck_len = spine_neck_len,
                     spine_head_diam = spine_head_diam,
                     soma_diameter = soma_diameter,
                     neck_dynamics = neck_dynamics,
                     space_clamp = space_clamp,
                     record_dendrite = record_dendrite, 
                     record_location = record_location, 
                     record_currents = record_currents,
                     record_branch = record_branch,
                     dend_glut2 = dend_glut2, 
                     record_mechs = record_mechs,
                     mechs2record = mechs2record,
                     record_path_dend = record_path_dend,   
                     record_path_spines = record_path_spines,  
                     record_all_path = record_all_path,
                     record_3D = record_3D,         
                     record_3D_impedance = record_3D_impedance,
                     freq = freq,                   
                     record_3D_mechs = record_3D_mechs,    
                     record_Ca = record_Ca,
                     record_3D_Ca = record_3D_Ca,
                     tonic = tonic,               
                     gbar_gaba = gbar_gaba,
                     rectification = rectification,       
                     distributed = distributed,         
                     gaba_params = gaba_params,
                     tonic_gaba_reversal = tonic_gaba_reversal,
                     dt = dt,
                     ds_imp = ds_imp,
                     voltage_clamp = voltage_clamp,
                     holding_potential = holding_potential,
                     voltage_clamp_site = voltage_clamp_site,
                     voltage_clamp_spine = voltage_clamp_spine,
                     voltage_clamp_loc = voltage_clamp_loc,
                     Rs = Rs,
                     downsample=downsample,
                     ds=ds
            )

            ind1 = 1
            ind2 = int(sim_time/deltat)
            t2 = np.arange(1, int(sim_time/deltat), 1) * deltat - deltat
            
            if voltage_clamp:
                # recorded currents are in nA so convert to pA
                iClamp.append(normalise(i_recording_site, stim_time, burn_time, deltat) * 1000)
                vdend_Clamp.append(normalise(vdend, stim_time, burn_time, deltat))
            else:
                soma_v_master.append(go.Scatter(x=t2, y=extract2(vsoma)[ind1:ind2]))
                dend_v_master.append(go.Scatter(x=t2, y=extract2(vdend)[ind1:ind2]))
            
            # only do if want to view each sim or save sim graphs
            if record_path_dend:
                plots_return(v_tree=v_dend_tree['v'], vspine=vspine, dists=v_dend_tree['dists'], spine_dist=spine_dist, 
                             num_gluts=num_gluts, start_time=start_time, burn_time=burn_time, dt=deltat, xaxis_range=[0,600], 
                             Nsim_plot=Nsim_plot, Nsim_save=Nsim_save, sim_image_path=sim_image_path, time=time)

            update_dictionary(data_dict=data_dict, protocol=protocol, i_recording_site=i_recording_site, v_recording_site=v_recording_site,
                    v_all_3D=v_all_3D, Ca_all_3D=Ca_all_3D, imp_all_3D=imp_all_3D, i_mechs_3D=i_mechs_3D, 
                    vspine=vspine, v_spine_activated=v_spine_activated, vdend=vdend, v_dend_activated=v_dend_activated, 
                    vsoma=vsoma, v_dend_tree=v_dend_tree, v_spine_tree=v_spine_tree, Ca_spine=Ca_spine, 
                    Ca_dend=Ca_dend, Ca_soma=Ca_soma, timing=timing, i_mechs_dend=i_mechs_dend, 
                    i_mechs_dend_tree=i_mechs_dend_tree, i_mechs_spine_tree=i_mechs_spine_tree, v_branch=v_branch, 
                    ampa_currents=ampa_currents, nmda_currents=nmda_currents, gaba_currents=gaba_currents, 
                    gaba_conductances=gaba_conductances, record_dist=record_dist, time=time, 
                    record_currents=record_currents, record_spine=record_spine, spine_dist=spine_dist, cap=cap)

        data_dict['metadata'] = metadata
        data_dict['metadata']['dt'] = dt
        data_dict['mechanisms_3D_distribution'] = mech_distr_3D

        data_dict['metadata']['record_location'] = record_location
        data_dict['metadata']['glutamate_locations'] = glutamate_locations

        # Save
        if save:
            simulations_path = 'downsample/{}/model{}/{}/simulations/sim{}/{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim_root, sim_letter)            
            os.makedirs(simulations_path, exist_ok=True)
            names = list(data_dict.keys())
            for name in names:
                with open('{}/{}.pickle'.format(simulations_path, name), 'wb') as handle:
                    pickle.dump(data_dict[name], handle, protocol=pickle.HIGHEST_PROTOCOL)  

        # generate plots
        time = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
        sims1 = ['1040', 1000, 1008, 1100] 
        if check_sim(sim, sims1):
            plt1(data_dict)
    


In [ ]:
if save: plotsave = save

plotsave = save

if voltage_clamp:
    out = max_and_sign(iClamp)
    x = np.arange(0, len(iClamp[0]), 1) * deltat
    if out[1] == 1:
        yaxis_range = [0, out[0]]
    else:
        yaxis_range = [out[0]*out[1], 0] 
        
    xaxis_range = [0, 600]
    x_err_bar=50

    fig1 = plot9(x=x, ydict=iClamp, yaxis='', xaxis='', yaxis_range=yaxis_range, xaxis_range=xaxis_range, y_err_bar=100, ybar_units='pA', x_err_bar=x_err_bar, y_err_bar_shift=10, width=800, height=400, offset=True, offset_ms=50) 
    fig1.show(config={"toImageButtonOptions": {"format": "svg"}})
    
    iClamp_norm = [
        i / np.max(np.abs(i)) if np.max(np.abs(i)) != 0 else i
        for i in iClamp
    ]
    fig2 = plot9(x=x, ydict=iClamp_norm, yaxis='', xaxis='', yaxis_range=[-1.05,0.05], xaxis_range=xaxis_range, y_err_bar=0, ybar_units='pA', x_err_bar=x_err_bar, y_err_bar_shift=0.05, width=800, height=400) 
    fig2.show(config={"toImageButtonOptions": {"format": "svg"}}) 

    fig3 = plot9(x=x, ydict=vdend_Clamp, yaxis='', xaxis='', yaxis_range=[0,40], xaxis_range=xaxis_range, y_err_bar=2, x_err_bar=x_err_bar, y_err_bar_shift=2, width=800, height=400, offset=True, offset_ms=50) 
    fig3.show(config={"toImageButtonOptions": {"format": "svg"}})

    if plotsave:
        sim_image_path = 'downsample/{}/model{}/{}/images/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim_root)
        os.makedirs(sim_image_path, exist_ok=True)
        fig1.write_image(f"{sim_image_path}/fig_soma_current.svg", format='svg', scale=3)
        fig2.write_image(f"{sim_image_path}/fig_soma_current_scaled.svg", format='svg', scale=3)
        fig3.write_image(f"{sim_image_path}/fig_dend_voltage_breakthrough.svg", format='svg', scale=3)
    
else:
    fig_soma_master, fig_dend_master = plot3(somaV=soma_v_master, dendV=dend_v_master, yaxis='V (mV)', stim_time = 90, sim_time=sim_time, black_trace=None, offset=True, offset_ms=200, x_err_bar=50, baseline=50, dt=deltat, ds=1)        
    fig_soma_master.show(config={"toImageButtonOptions": {"format": "svg"}})
    fig_dend_master.show(config={"toImageButtonOptions": {"format": "svg"}})
    if plotsave:
        sim_image_path = 'downsample/{}/model{}/{}/images/sim{}'.format(cell_type, model, 'physiological' if physiological else 'nonphysiological', sim_root)
        os.makedirs(sim_image_path, exist_ok=True)
        fig_soma_master.write_image(f"{sim_image_path}/fig_soma_master.svg", format='svg', scale=3)
        fig_dend_master.write_image(f"{sim_image_path}/fig_dend_master.svg", format='svg', scale=3)


In [ ]:
[1e3 * g_GABA * v for k, v in N_GABA_dict.items()] # sum of conductances in nS (g_GABA in uS)

In [ ]:
N_GABA_dict 